# Feature Engineering M5 Forecasting Accuracy

Notebook ini mengimplementasikan **Tahap 5: Feature Engineering** sesuai Bab 3 skripsi.

## Prinsip metodologis
- Semua fitur dibangun dengan prinsip **no lookahead leakage**.
- `lag features` dihitung menggunakan `shift(k)` sehingga hanya memakai informasi masa lalu.
- `rolling statistics` dihitung dari `shift(1)` lalu `rolling(window)` agar nilai target hari yang sedang diprediksi tidak ikut masuk ke fitur.
- Statistik harga historis seperti `price_mean` dihitung hanya dari **training set utama** lalu diterapkan ke test split.
- Fitur grup (`item`, `dept`, `cat`, `store`, `state`, `store_cat`, `store_dept`) dibawa dari hasil preprocessing sebagai konteks hierarkis.

## Fitur yang dibangun
- Lag features: `7, 14, 21, 28, 35, 42, 49, 56, 63, 70, 77, 84, 90, 180, 364`
- Rolling statistics: `mean`, `std`, `max`, `min` untuk window `7, 14, 28, 60, 90, 180, 364`
- Calendar features: `week`, `day`, `days_to_event`
- Price features: `price_mean`, `price_norm`, `price_cat_avg`, `price_cat_ratio`, `price_change`, `price_changed`, `is_discounted`


In [11]:
from __future__ import annotations

from pathlib import Path
import json

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)


In [12]:
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATASET_DIR = PROJECT_ROOT / "dataset"
PREPROCESSED_DIR = DATASET_DIR / "preprocessed"
FEATURE_DIR = DATASET_DIR / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

MODE = "smoke"  # ganti menjadi "full" setelah preprocessing full tersedia lengkap
COMPRESSION = "gzip"
LAG_DAYS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70, 77, 84, 90, 180, 364]
ROLLING_WINDOWS = [7, 14, 28, 60, 90, 180, 364]
ROLLING_STATS = ["mean", "std", "max", "min"]

TRAIN_INPUT_PATH = PREPROCESSED_DIR / f"train_preprocessed_{MODE}.csv.gz"
TEST_INPUT_PATH = PREPROCESSED_DIR / f"test_preprocessed_{MODE}.csv.gz"
PREPROCESSING_MAPPINGS_PATH = PREPROCESSED_DIR / f"preprocessing_mappings_{MODE}.json"
PREPROCESSING_SUMMARY_PATH = PREPROCESSED_DIR / f"preprocessing_summary_{MODE}.json"

TRAIN_OUTPUT_PATH = FEATURE_DIR / f"train_features_{MODE}.csv.gz"
TEST_OUTPUT_PATH = FEATURE_DIR / f"test_features_{MODE}.csv.gz"
SUMMARY_OUTPUT_PATH = FEATURE_DIR / f"feature_engineering_summary_{MODE}.json"

assert TRAIN_INPUT_PATH.exists(), f"Input train preprocessing tidak ditemukan: {TRAIN_INPUT_PATH}"
assert TEST_INPUT_PATH.exists(), f"Input test preprocessing tidak ditemukan: {TEST_INPUT_PATH}"

print(f"PROJECT_ROOT:      {PROJECT_ROOT}")
print(f"MODE:              {MODE}")
print(f"TRAIN_INPUT_PATH:  {TRAIN_INPUT_PATH}")
print(f"TEST_INPUT_PATH:   {TEST_INPUT_PATH}")
print(f"TRAIN_OUTPUT_PATH: {TRAIN_OUTPUT_PATH}")
print(f"TEST_OUTPUT_PATH:  {TEST_OUTPUT_PATH}")


PROJECT_ROOT:      C:\Users\adeli\speedrun-lulus
MODE:              smoke
TRAIN_INPUT_PATH:  C:\Users\adeli\speedrun-lulus\dataset\preprocessed\train_preprocessed_smoke.csv.gz
TEST_INPUT_PATH:   C:\Users\adeli\speedrun-lulus\dataset\preprocessed\test_preprocessed_smoke.csv.gz
TRAIN_OUTPUT_PATH: C:\Users\adeli\speedrun-lulus\dataset\features\train_features_smoke.csv.gz
TEST_OUTPUT_PATH:  C:\Users\adeli\speedrun-lulus\dataset\features\test_features_smoke.csv.gz


## 1. Load hasil preprocessing

Tahap feature engineering menggunakan output dari tahap preprocessing, bukan data mentah langsung.


In [13]:
train_df = pd.read_csv(TRAIN_INPUT_PATH, compression=COMPRESSION)
test_df = pd.read_csv(TEST_INPUT_PATH, compression=COMPRESSION)

with open(PREPROCESSING_MAPPINGS_PATH, "r", encoding="utf-8") as f:
    preprocessing_mappings = json.load(f)

with open(PREPROCESSING_SUMMARY_PATH, "r", encoding="utf-8") as f:
    preprocessing_summary = json.load(f)

print("train_df shape:", train_df.shape)
print("test_df shape: ", test_df.shape)
train_df.head(3)


train_df shape: (9565, 49)
test_df shape:  (140, 49)


,id,item_id,dept_id,cat_id,store_id,state_id,store_cat,store_dept,date,d,wm_yr_wk,weekday,wday,dayofweek,month,year,is_weekend,event_name_1,event_name_2,event_type_1,event_type_2,event_name_1_le,event_name_2_le,snap,sell_price,sales,split,item_id_le,dept_id_le,cat_id_le,store_id_le,state_id_le,store_cat_le,store_dept_le,item_id_mean_enc,dept_id_mean_enc,cat_id_mean_enc,store_id_mean_enc,state_id_mean_enc,store_cat_mean_enc,store_dept_mean_enc,event_type_1__Cultural,event_type_1__National,event_type_1__No_Type,event_type_1__Religious,event_type_1__Sporting,event_type_2__Cultural,event_type_2__No_Type,event_type_2__Religious
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,CA_1|HOBBIES,CA_1|HOBBIES_1,2011-01-29,d_1,11101,Saturday,1,5,1,2011,1,No_Event,No_Event,No_Type,No_Type,19,3,0,9.58,0,train,0,0,0,0,0,0,0,0.314602,0.681443,0.681443,0.681443,0.681443,0.681443,0.681443,0,0,1,0,0,0,1,0
1,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,CA_1|HOBBIES,CA_1|HOBBIES_1,2011-01-30,d_2,11101,Sunday,2,6,1,2011,1,No_Event,No_Event,No_Type,No_Type,19,3,0,9.58,0,train,0,0,0,0,0,0,0,0.314602,0.681443,0.681443,0.681443,0.681443,0.681443,0.681443,0,0,1,0,0,0,1,0
2,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,CA_1|HOBBIES,CA_1|HOBBIES_1,2011-01-31,d_3,11101,Monday,3,0,1,2011,0,No_Event,No_Event,No_Type,No_Type,19,3,0,9.58,0,train,0,0,0,0,0,0,0,0.314602,0.681443,0.681443,0.681443,0.681443,0.681443,0.681443,0,0,1,0,0,0,1,0


## 2. Helper functions


In [14]:
def cast_numeric_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"])
    numeric_int_cols = [
        "wm_yr_wk", "wday", "dayofweek", "month", "year", "is_weekend", "snap", "sales"
    ]
    for col in numeric_int_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    out["sell_price"] = pd.to_numeric(out["sell_price"], errors="coerce")
    return out


def add_calendar_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["week"] = out["date"].dt.isocalendar().week.astype("int32")
    out["day"] = out["date"].dt.day.astype("int16")

    event_mask = (out["event_name_1"] != "No_Event") | (out["event_name_2"] != "No_Event")
    all_event_dates = np.sort(out.loc[event_mask, "date"].drop_duplicates().to_numpy(dtype="datetime64[ns]"))
    if len(all_event_dates) == 0:
        out["days_to_event"] = 999
        return out

    current_dates = out["date"].to_numpy(dtype="datetime64[ns]")
    insertion_idx = np.searchsorted(all_event_dates, current_dates, side="left")
    next_event_dates = np.where(
        insertion_idx < len(all_event_dates),
        all_event_dates[np.clip(insertion_idx, 0, len(all_event_dates) - 1)],
        current_dates,
    )
    days_to_event = ((next_event_dates - current_dates) / np.timedelta64(1, "D")).astype("int32")
    days_to_event[insertion_idx >= len(all_event_dates)] = 999
    out["days_to_event"] = days_to_event
    return out


def add_price_features(df: pd.DataFrame, train_reference: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    out = df.copy()

    price_mean_map = (
        train_reference.groupby(["store_id", "item_id"], dropna=False)["sell_price"]
        .mean()
        .rename("price_mean")
        .reset_index()
    )
    out = out.merge(price_mean_map, on=["store_id", "item_id"], how="left", validate="many_to_one")

    global_price_mean = float(train_reference["sell_price"].mean())
    out["price_mean"] = out["price_mean"].fillna(global_price_mean)
    out["price_norm"] = out["sell_price"] / out["price_mean"].replace(0, np.nan)

    out["price_cat_avg"] = (
        out.groupby(["store_id", "cat_id", "wm_yr_wk"], dropna=False)["sell_price"]
        .transform("mean")
    )
    out["price_cat_ratio"] = out["sell_price"] / out["price_cat_avg"].replace(0, np.nan)

    out = out.sort_values(["store_id", "item_id", "date"]).reset_index(drop=True)
    out["price_change"] = out.groupby(["store_id", "item_id"], sort=False)["sell_price"].diff(1)
    out["price_changed"] = out["price_change"].fillna(0).ne(0).astype("int8")
    out["is_discounted"] = out["sell_price"].lt(out["price_mean"]).astype("int8")

    fill_zero_cols = ["price_norm", "price_cat_ratio", "price_change"]
    for col in fill_zero_cols:
        out[col] = out[col].replace([np.inf, -np.inf], np.nan).fillna(0)

    artifact = {
        "global_price_mean": global_price_mean,
        "price_reference_pairs": int(len(price_mean_map)),
    }
    return out, artifact


def add_lag_features(df: pd.DataFrame, lag_days: list[int]) -> pd.DataFrame:
    out = df.copy()
    grouped_sales = out.groupby("id", sort=False)["sales"]
    for lag in lag_days:
        out[f"lag_{lag}"] = grouped_sales.shift(lag).fillna(0)
    return out


def add_rolling_features(df: pd.DataFrame, windows: list[int]) -> pd.DataFrame:
    out = df.copy()
    grouped_sales = out.groupby("id", sort=False)["sales"]
    shifted = grouped_sales.shift(1)

    for window in windows:
        roller = shifted.groupby(out["id"], sort=False).rolling(window=window, min_periods=1)
        out[f"rolling_mean_{window}"] = roller.mean().reset_index(level=0, drop=True)
        out[f"rolling_std_{window}"] = roller.std().reset_index(level=0, drop=True).fillna(0)
        out[f"rolling_max_{window}"] = roller.max().reset_index(level=0, drop=True)
        out[f"rolling_min_{window}"] = roller.min().reset_index(level=0, drop=True)

    rolling_cols = [col for col in out.columns if col.startswith("rolling_")]
    out[rolling_cols] = out[rolling_cols].fillna(0)
    return out


def downcast_feature_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in out.columns:
        if col.startswith("lag_") or col.startswith("rolling_") or col.startswith("price_"):
            out[col] = pd.to_numeric(out[col], downcast="float")
    if "days_to_event" in out.columns:
        out["days_to_event"] = pd.to_numeric(out["days_to_event"], downcast="integer")
    if "week" in out.columns:
        out["week"] = pd.to_numeric(out["week"], downcast="integer")
    if "day" in out.columns:
        out["day"] = pd.to_numeric(out["day"], downcast="integer")
    return out


## 3. Gabungkan train dan test untuk konstruksi fitur temporal

Penggabungan dilakukan agar baris pada `test_public` tetap dapat memakai histori dari `train` untuk fitur `lag` dan `rolling`, sambil tetap menjaga `shift` agar tidak ada target hari yang sama masuk ke fitur.


In [15]:
train_df = cast_numeric_columns(train_df)
test_df = cast_numeric_columns(test_df)

combined_df = pd.concat([train_df, test_df], axis=0, ignore_index=True)
combined_df = combined_df.sort_values(["id", "date"]).reset_index(drop=True)

assert combined_df[["id", "date"]].duplicated().sum() == 0
combined_df[["id", "date", "sales", "split"]].head(5)


,id,date,sales,split
0,HOBBIES_1_001_CA_1_evaluation,2011-01-29,0,train
1,HOBBIES_1_001_CA_1_evaluation,2011-01-30,0,train
2,HOBBIES_1_001_CA_1_evaluation,2011-01-31,0,train
3,HOBBIES_1_001_CA_1_evaluation,2011-02-01,0,train
4,HOBBIES_1_001_CA_1_evaluation,2011-02-02,0,train


## 4. Bangun calendar features dan price features


In [16]:
combined_df = add_calendar_features(combined_df)
combined_df, price_artifact = add_price_features(combined_df, train_reference=train_df)
combined_df[["date", "week", "day", "days_to_event", "sell_price", "price_mean", "price_norm", "price_change"]].head(10)


,date,week,day,days_to_event,sell_price,price_mean,price_norm,price_change
0,2011-01-29,4,29,8,9.58,8.888918,1.077746,0.0
1,2011-01-30,4,30,7,9.58,8.888918,1.077746,0.0
2,2011-01-31,5,31,6,9.58,8.888918,1.077746,0.0
3,2011-02-01,5,1,5,9.58,8.888918,1.077746,0.0
4,2011-02-02,5,2,4,9.58,8.888918,1.077746,0.0
5,2011-02-03,5,3,3,9.58,8.888918,1.077746,0.0
6,2011-02-04,5,4,2,9.58,8.888918,1.077746,0.0
7,2011-02-05,5,5,1,9.58,8.888918,1.077746,0.0
8,2011-02-06,5,6,0,9.58,8.888918,1.077746,0.0
9,2011-02-07,6,7,7,9.58,8.888918,1.077746,0.0


## 5. Bangun lag features dan rolling statistics

Semua fitur temporal dibangun dengan `shift` sesuai metodologi Bab 3.


In [17]:
combined_df = add_lag_features(combined_df, LAG_DAYS)
combined_df = add_rolling_features(combined_df, ROLLING_WINDOWS)
combined_df = downcast_feature_columns(combined_df)

feature_cols_preview = [
    "sales", "lag_7", "lag_28", "lag_364", "rolling_mean_7", "rolling_std_28", "rolling_max_90", "rolling_min_180"
]
combined_df[["id", "date", "split"] + feature_cols_preview].head(10)


,id,date,split,sales,lag_7,lag_28,lag_364,rolling_mean_7,rolling_std_28,rolling_max_90,rolling_min_180
0,HOBBIES_1_001_CA_1_evaluation,2011-01-29,train,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,HOBBIES_1_001_CA_1_evaluation,2011-01-30,train,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,HOBBIES_1_001_CA_1_evaluation,2011-01-31,train,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,HOBBIES_1_001_CA_1_evaluation,2011-02-01,train,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,HOBBIES_1_001_CA_1_evaluation,2011-02-02,train,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,HOBBIES_1_001_CA_1_evaluation,2011-02-03,train,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,HOBBIES_1_001_CA_1_evaluation,2011-02-04,train,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
7,HOBBIES_1_001_CA_1_evaluation,2011-02-05,train,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,HOBBIES_1_001_CA_1_evaluation,2011-02-06,train,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,HOBBIES_1_001_CA_1_evaluation,2011-02-07,train,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 6. Leakage sanity checks

Validasi sederhana untuk memastikan fitur temporal mengikuti prinsip no-lookahead.


In [18]:
first_series = combined_df[combined_df["id"] == combined_df["id"].iloc[0]].copy().sort_values("date")
assert (first_series.iloc[:7]["lag_7"] == 0).all(), "lag_7 pada burn-in period harus nol"
assert first_series.iloc[0]["rolling_mean_7"] == 0, "rolling_mean_7 awal harus nol"

train_only_for_corr = combined_df[combined_df["split"] == "train"].copy()
corr_same_day = train_only_for_corr[["sales", "rolling_mean_7"]].corr().iloc[0, 1]
corr_prev_day = train_only_for_corr[["lag_7", "rolling_mean_7"]].corr().iloc[0, 1]

print("corr(sales, rolling_mean_7):", corr_same_day)
print("corr(lag_7, rolling_mean_7):", corr_prev_day)


corr(sales, rolling_mean_7): 0.47491180660364224
corr(lag_7, rolling_mean_7): 0.6245798505327697


## 7. Pisahkan kembali train dan test lalu simpan hasil


In [19]:
train_features = combined_df[combined_df["split"] == "train"].copy()
test_features = combined_df[combined_df["split"] == "test_public"].copy()

train_features.to_csv(TRAIN_OUTPUT_PATH, index=False, compression=COMPRESSION)
test_features.to_csv(TEST_OUTPUT_PATH, index=False, compression=COMPRESSION)

feature_summary = {
    "source": "Outline Skripsi Bab 3 - Tahap Feature Engineering",
    "mode": MODE,
    "compression": COMPRESSION,
    "lag_days": LAG_DAYS,
    "rolling_windows": ROLLING_WINDOWS,
    "rolling_statistics": ROLLING_STATS,
    "notes": [
        "Lag features dihitung dengan shift(k) sehingga hanya memakai informasi masa lalu.",
        "Rolling statistics dihitung dari shift(1) sebelum rolling untuk mencegah lookahead leakage.",
        "Price mean dihitung hanya dari training set utama.",
        "Group features dipertahankan dari hasil preprocessing.",
    ],
    "price_artifact": price_artifact,
    "train_result": {
        "row_count": int(len(train_features)),
        "column_count": int(train_features.shape[1]),
        "output_path": str(TRAIN_OUTPUT_PATH.resolve()),
    },
    "test_result": {
        "row_count": int(len(test_features)),
        "column_count": int(test_features.shape[1]),
        "output_path": str(TEST_OUTPUT_PATH.resolve()),
    },
}

with open(SUMMARY_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(feature_summary, f, indent=2)

feature_summary


{'source': 'Outline Skripsi Bab 3 - Tahap Feature Engineering',
 'mode': 'smoke',
 'compression': 'gzip',
 'lag_days': [7, 14, 21, 28, 35, 42, 49, 56, 63, 70, 77, 84, 90, 180, 364],
 'rolling_windows': [7, 14, 28, 60, 90, 180, 364],
 'rolling_statistics': ['mean', 'std', 'max', 'min'],
 'notes': ['Lag features dihitung dengan shift(k) sehingga hanya memakai informasi masa lalu.',
  'Rolling statistics dihitung dari shift(1) sebelum rolling untuk mencegah lookahead leakage.',
  'Price mean dihitung hanya dari training set utama.',
  'Group features dipertahankan dari hasil preprocessing.'],
 'price_artifact': {'global_price_mean': 4.659098797699947,
  'price_reference_pairs': 5},
 'train_result': {'row_count': 9565,
  'column_count': 102,
  'output_path': 'C:\\Users\\adeli\\speedrun-lulus\\dataset\\features\\train_features_smoke.csv.gz'},
 'test_result': {'row_count': 140,
  'column_count': 102,
  'output_path': 'C:\\Users\\adeli\\speedrun-lulus\\dataset\\features\\test_features_smoke.c

## 8. Preview hasil akhir


In [20]:
train_preview = pd.read_csv(TRAIN_OUTPUT_PATH, compression=COMPRESSION, nrows=5)
test_preview = pd.read_csv(TEST_OUTPUT_PATH, compression=COMPRESSION, nrows=5)

print("train feature shape preview:", train_preview.shape)
print("test feature shape preview: ", test_preview.shape)
train_preview[[
    "id", "date", "sales", "lag_7", "lag_28", "rolling_mean_7", "rolling_std_28", "price_norm", "days_to_event"
]].head()


train feature shape preview: (5, 102)
test feature shape preview:  (5, 102)


,id,date,sales,lag_7,lag_28,rolling_mean_7,rolling_std_28,price_norm,days_to_event
0,HOBBIES_1_001_CA_1_evaluation,2011-01-29,0,0.0,0.0,0.0,0.0,1.077746,8
1,HOBBIES_1_001_CA_1_evaluation,2011-01-30,0,0.0,0.0,0.0,0.0,1.077746,7
2,HOBBIES_1_001_CA_1_evaluation,2011-01-31,0,0.0,0.0,0.0,0.0,1.077746,6
3,HOBBIES_1_001_CA_1_evaluation,2011-02-01,0,0.0,0.0,0.0,0.0,1.077746,5
4,HOBBIES_1_001_CA_1_evaluation,2011-02-02,0,0.0,0.0,0.0,0.0,1.077746,4
